# MCP Server Skeleton: Smoke Test

Covers task backlog §3.1. `mcp_server.py` is a long-running process (`.py` per the
file-format convention, reason (a)), so it can't be re-executed top-to-bottom the way a
notebook is -- this notebook is the verification artifact instead: it actually **spawns
`mcp_server.py` as a real subprocess** and talks to it over the real stdio transport
(via the MCP SDK's own client), rather than importing the server object in-process. That
matters because stdio subprocess spawning is exactly how the real Retriever node will
eventually talk to this server -- this test exercises the actual integration point, not
a shortcut around it.

**Transport: stdio, not streamable HTTP/SSE.** This server is spawned locally by
whatever process runs the LangGraph app -- not a shared network service multiple
independent clients connect to over a network. stdio needs no port/network config and
is the standard pattern for a locally-spawned MCP server (same model Claude Desktop uses).
SSE/streamable-http remain available later via `mcp_server.run(transport=...)` without
restructuring the tools themselves, if this project ever needs the server reachable
from a separate process/machine.

**Both tools are real stubs** (§3.2/§3.3 not built yet, sanctioned by the CP1 milestone
checklist: "MCP server skeleton + tool schemas, implementation can still be stubbed").
This notebook checks two things: the schemas are real (tool discovery works), and
calling a stub tool comes back as a clean, honest error -- not a crash, not a silent
fake success.

In [1]:
# !pip install mcp

## Setup

In [2]:
import os
import sys

from mcp.client.session import ClientSession
from mcp.client.stdio import StdioServerParameters, stdio_client

SERVER_PATH = os.path.join(os.getcwd(), "mcp_server.py")
assert os.path.exists(SERVER_PATH), f"mcp_server.py not found at {SERVER_PATH}"

server_params = StdioServerParameters(command=sys.executable, args=[SERVER_PATH])
print(f"Will spawn: {sys.executable} {SERVER_PATH}")

Will spawn: /opt/anaconda3/bin/python /Users/pareeyaporn/Desktop/ADSP 32028 Applied Generative AI - Agents and Multimodal Intelligence/Project/Code/3. MCP Server & Tools/mcp_server.py


## Tool discovery

Real subprocess spawn + real stdio handshake (`session.initialize()`) + real
`list_tools()` call -- not a mock, not an in-process shortcut.

In [3]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools_result = await session.list_tools()

        tool_names = [t.name for t in tools_result.tools]
        print("Discovered tools:", tool_names)
        for t in tools_result.tools:
            print(f"\n[{t.name}] {t.description}")
            print("  input_schema:", t.input_schema)

        assert set(tool_names) == {"rag.search", "web.search"}, "expected exactly these two tools"
        rag_schema = next(t.input_schema for t in tools_result.tools if t.name == "rag.search")
        assert rag_schema["required"] == ["query"], "query should be the only required field"
        assert set(rag_schema["properties"]) == {"query", "top_k", "budget_min", "budget_max", "category", "brand"}
        print("\n[OK] both tools discovered with the expected real schemas.")

Discovered tools: ['rag.search', 'web.search']

[rag.search] Search the private product catalog (Home & Kitchen demo slice).
  input_schema: {'properties': {'query': {'title': 'Query', 'type': 'string'}, 'top_k': {'default': 5, 'title': 'Top K', 'type': 'integer'}, 'budget_min': {'anyOf': [{'type': 'number'}, {'type': 'null'}], 'default': None, 'title': 'Budget Min'}, 'budget_max': {'anyOf': [{'type': 'number'}, {'type': 'null'}], 'default': None, 'title': 'Budget Max'}, 'category': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Category'}, 'brand': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Brand'}}, 'required': ['query'], 'type': 'object', 'title': 'rag_searchArguments'}

[web.search] Search the live web for current pricing/availability (Tavily).
  input_schema: {'properties': {'query': {'title': 'Query', 'type': 'string'}, 'top_k': {'default': 5, 'title': 'Top K', 'type': 'integer'}}, 'required': ['query'], 'type': 'obj

## Confirming server liveness

Both tools are real now (`rag.search` since §3.2, `web.search` since §3.3), each with
their own dedicated, deeper test notebook. This section only confirms the server
responds successfully post-wiring -- calling `rag.search` again here (free: local
embedding model + Chroma, no external API). Deliberately **not** re-calling `web.search`
here -- that would spend a real Tavily credit redundantly on top of what §3.3's own
notebook already spent proving the same thing.

In [4]:
async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        rag_result = await session.call_tool("rag.search", {"query": "insulated lunch box"})
        print("rag.search call -- is_error:", rag_result.is_error)

        assert rag_result.is_error is False, "rag.search should succeed (real, §3.2)"
        print("\n[OK] server responds correctly post-wiring. web.search verified separately in §3.3's own notebook (no redundant Tavily spend here).")

rag.search call -- is_error: False

[OK] server responds correctly post-wiring. web.search verified separately in §3.3's own notebook (no redundant Tavily spend here).


## Next steps

- §3.2: replace `rag_search`'s body with a real call into §2.4's `retriever.py`
  `search()` (already ported once from §1.4 -- this would be the second, final home for
  that logic, matching §2.4's own note about a future cleanup)
- §3.3: replace `web_search`'s body with a real Tavily call (provider already decided)
- Once §3.2/§3.3 land, §2.4's Retriever should swap from calling `search()` directly to
  calling these tools over MCP (the one remaining unchecked item from §2.4)